# Notebook 2 - Feature Engineering


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
OUTPUT_PATH = Path(r"path/to/your/data")

# Load panels from notebook 1
bookings_daily = pl.read_parquet(OUTPUT_PATH / "bookings_daily.parquet")
reddit_daily   = pl.read_parquet(OUTPUT_PATH / "reddit_daily.parquet")

print("Bookings panel :", bookings_daily.shape)
print("Reddit panel   :", reddit_daily.shape)
print()
print("Bookings date range:", bookings_daily["date"].min(), "→", bookings_daily["date"].max())
print("Reddit date range  :", reddit_daily["date"].min(),   "→", reddit_daily["date"].max())

## Fill missing dates per country

In [ ]:
def fill_date_spine(df: pl.DataFrame, date_col: str, group_col: str, fill_value: dict) -> pl.DataFrame:
    """
    For each group (country), ensure every date in the global range exists.
    Missing dates are filled with fill_value.
    """
    date_min = df[date_col].min()
    date_max = df[date_col].max()
    
    # Full date spine
    all_dates = pl.date_range(date_min, date_max, interval="1d", eager=True)
    groups    = df[group_col].unique()
    
    # Cross join: every country × every date
    spine = pl.DataFrame({date_col: all_dates}).join(pl.DataFrame({group_col: groups}), how="cross")
    
    # Left join original data onto spine, fill nulls
    filled = spine.join(df, on=[date_col, group_col], how="left")
    for col, val in fill_value.items():
        filled = filled.with_columns(pl.col(col).fill_null(val))
    
    return filled.sort([group_col, date_col])


# Fill bookings
bookings_filled = fill_date_spine(
    bookings_daily,
    date_col="date",
    group_col="lhg_country",
    fill_value={"pax": 0}
)

# Fill reddit
reddit_filled = fill_date_spine(
    reddit_daily,
    date_col="date",
    group_col="lhg_country",
    fill_value={
        "total_engagement":        0,
        "total_engagement_score":  0.0,
        "total_weighted_sentiment":0.0,
        "mean_sentiment":          0.0,
        "n_posts":                 0,
    }
)

print(f"Bookings after fill: {bookings_filled.shape}")
print(f"Reddit after fill  : {reddit_filled.shape}")

## Booking-side features

In [ ]:
# Rolling window aggregations
# We use Polars group_by_dynamic for per-country rolling windows

WINDOWS = [7, 14, 28]  # days

def add_rolling_features(df: pl.DataFrame, value_col: str, group_col: str, windows: list) -> pl.DataFrame:
    """Add rolling sum and mean for each window size, per group."""
    result = df.sort([group_col, "date"])
    for w in windows:
        result = result.with_columns([
            pl.col(value_col)
              .rolling_sum(window_size=w, min_periods=1)
              .over(group_col)
              .alias(f"{value_col}_roll{w}sum"),
            pl.col(value_col)
              .rolling_mean(window_size=w, min_periods=1)
              .over(group_col)
              .alias(f"{value_col}_roll{w}mean"),
        ])
    return result

bookings_feat = add_rolling_features(bookings_filled, "pax", "lhg_country", WINDOWS)

print("Booking features added:")
print([c for c in bookings_feat.columns if "roll" in c])

In [ ]:
# Per-country spike detection 
# Same z-score logic as notebook 01 but applied per country.
# This is the target variable for the correlation analysis:
# "did this country experience an unexpected booking surge today?"

SPIKE_WINDOW    = 28   # rolling baseline window
SPIKE_THRESHOLD = 2.0  # |z| > 2 = anomalous

bookings_feat = bookings_feat.with_columns([
    # Rolling mean and std per country
    pl.col("pax")
      .rolling_mean(window_size=SPIKE_WINDOW, min_periods=7)
      .over("lhg_country")
      .alias("pax_roll28_baseline"),
    pl.col("pax")
      .rolling_std(window_size=SPIKE_WINDOW, min_periods=7)
      .over("lhg_country")
      .alias("pax_roll28_std"),
]).with_columns([
    # Z-score
    ((pl.col("pax") - pl.col("pax_roll28_baseline")) /
     (pl.col("pax_roll28_std") + 1e-8))   # +epsilon avoids div/0
    .alias("pax_zscore"),
]).with_columns([
    # Binary spike flag
    (pl.col("pax_zscore").abs() > SPIKE_THRESHOLD)
    .cast(pl.Int8)
    .alias("pax_spike"),
    # Positive spike only (demand surge, not drop)
    (pl.col("pax_zscore") > SPIKE_THRESHOLD)
    .cast(pl.Int8)
    .alias("pax_spike_positive"),
])

n_spikes = bookings_feat["pax_spike_positive"].sum()
print(f"Total positive booking spikes across all countries: {n_spikes}")
print()

# Top countries by spike frequency
spike_by_country = (
    bookings_feat
    .group_by("lhg_country")
    .agg(pl.col("pax_spike_positive").sum().alias("n_spikes"))
    .sort("n_spikes", descending=True)
    .head(15)
)
print("Countries with most booking spikes:")
print(spike_by_country)

In [ ]:
# Visualise spikes for a sample country
# Change SAMPLE_COUNTRY to any alpha-2 code from data
SAMPLE_COUNTRY = spike_by_country["lhg_country"][0]  # auto-picks most spiked country

sample = (
    bookings_feat
    .filter(pl.col("lhg_country") == SAMPLE_COUNTRY)
    .to_pandas()
    .set_index("date")
)

spike_dates = sample[sample["pax_spike_positive"] == 1].index

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(sample["pax"], linewidth=0.8, alpha=0.6, label="Daily pax")
axes[0].plot(sample["pax_roll28_baseline"], linewidth=1.5, color="orange", label="28d baseline")
for d in spike_dates:
    axes[0].axvline(d, color="crimson", alpha=0.3, linewidth=0.8)
axes[0].scatter(spike_dates, sample.loc[spike_dates, "pax"],
                color="crimson", s=25, zorder=5, label="Spike")
# axes[0].set_title(f"{SAMPLE_COUNTRY} — Daily bookings with spike detection")
axes[0].set_title(f"Daily bookings with spike detection")
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[1].plot(sample["pax_zscore"], linewidth=0.8, color="steelblue")
axes[1].axhline( SPIKE_THRESHOLD, color="crimson", linestyle="--", linewidth=1)
axes[1].axhline(-SPIKE_THRESHOLD, color="crimson", linestyle="--", linewidth=1)
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].set_title("Z-score (28d rolling)")

plt.tight_layout()
plt.savefig(OUTPUT_PATH / f"spikes_{SAMPLE_COUNTRY}.png", dpi=150)
plt.show()
print(f"Spike dates for {SAMPLE_COUNTRY}: {list(spike_dates.astype(str))}")

## Reddit-side features

### a. Rolling windows on engagement and sentiment

In [ ]:
reddit_feat = reddit_filled.sort(["lhg_country", "date"])

# Rolling windows on engagement score and weighted sentiment
for w in WINDOWS:
    reddit_feat = reddit_feat.with_columns([
        pl.col("total_engagement_score")
          .rolling_sum(window_size=w, min_periods=1)
          .over("lhg_country")
          .alias(f"eng_score_roll{w}sum"),
        pl.col("total_engagement_score")
          .rolling_mean(window_size=w, min_periods=1)
          .over("lhg_country")
          .alias(f"eng_score_roll{w}mean"),
        pl.col("total_weighted_sentiment")
          .rolling_sum(window_size=w, min_periods=1)
          .over("lhg_country")
          .alias(f"weighted_sent_roll{w}sum"),
        pl.col("mean_sentiment")
          .rolling_mean(window_size=w, min_periods=1)
          .over("lhg_country")
          .alias(f"mean_sent_roll{w}mean"),
    ])

print("Reddit rolling features added:")
print([c for c in reddit_feat.columns if "roll" in c])

### b. Sentiment momentum

Raw sentiment level is almost always positive for travel destinations.
What matters more is **whether sentiment is improving** — a destination
going from neutral to positive is a stronger signal than one that's
been consistently positive for months.

Momentum = 7-day rolling mean sentiment minus 28-day rolling mean sentiment.
Positive momentum = sentiment improving recently relative to the longer baseline.

In [ ]:
reddit_feat = reddit_feat.with_columns([
    # Sentiment momentum: short-term vs long-term average
    (pl.col("mean_sent_roll7mean") - pl.col("mean_sent_roll28mean"))
    .alias("sentiment_momentum"),

    # Engagement momentum: is buzz growing?
    (pl.col("eng_score_roll7mean") - pl.col("eng_score_roll28mean"))
    .alias("engagement_momentum"),

    # Engagement z-score per country (is today unusually buzzy?)
    (
        (pl.col("total_engagement_score") -
         pl.col("total_engagement_score").rolling_mean(window_size=28, min_periods=7).over("lhg_country")) /
        (pl.col("total_engagement_score").rolling_std(window_size=28, min_periods=7).over("lhg_country") + 1e-8)
    ).alias("engagement_zscore"),
])

print("Sentiment & engagement momentum features added.")
reddit_feat.select(["date", "lhg_country", "mean_sentiment",
                    "mean_sent_roll7mean", "mean_sent_roll28mean",
                    "sentiment_momentum", "engagement_momentum",
                    "engagement_zscore"]).head(10)

### c. Consistent sentiment feature


**Sentiment** = the average sentiment score over a window

**Consistent sentiment** = how stable and persistently positive sentiment has been.


In [ ]:
SENTIMENT_POSITIVE_THRESHOLD = 0.05  # mean_sentiment above this = positive day

reddit_feat = reddit_feat.with_columns([
    # Rolling standard deviation of sentiment (lower = more consistent)
    pl.col("mean_sentiment")
      .rolling_std(window_size=28, min_periods=7)
      .over("lhg_country")
      .alias("sentiment_consistency"),

    # Is today a positive sentiment day?
    (pl.col("mean_sentiment") > SENTIMENT_POSITIVE_THRESHOLD)
    .cast(pl.Int8)
    .alias("is_positive_day"),
]).with_columns([
    # Count of positive days in last 7 / 14 / 28 days
    pl.col("is_positive_day")
      .rolling_sum(window_size=7, min_periods=1)
      .over("lhg_country")
      .alias("positive_days_7d"),
    pl.col("is_positive_day")
      .rolling_sum(window_size=14, min_periods=1)
      .over("lhg_country")
      .alias("positive_days_14d"),
    pl.col("is_positive_day")
      .rolling_sum(window_size=28, min_periods=1)
      .over("lhg_country")
      .alias("positive_days_28d"),
]).with_columns([
    # Consistent sentiment score: proportion of positive days × mean sentiment level
    ((pl.col("positive_days_28d") / 28.0) * pl.col("mean_sent_roll28mean"))
    .alias("consistent_sentiment_score"),
]).with_columns([
    # Detrended positive days: above-baseline positivity streaks only
    # Subtracts the rolling mean so summer seasonality doesn't inflate the score
    (pl.col("positive_days_14d") -
     pl.col("positive_days_14d")
       .rolling_mean(window_size=28, min_periods=7)
       .over("lhg_country"))
    .alias("positive_days_14d_detrended"),
])

print("Consistent sentiment features added.")
reddit_feat.select(["date", "lhg_country", "mean_sentiment",
                    "sentiment_consistency", "positive_days_28d",
                    "consistent_sentiment_score",
                    "positive_days_14d_detrended"]).head(10)

### d. Lag features

We shift Reddit features forward in time by 7/14/21/28 days.
This means: the Reddit signal from 14 days ago is placed alongside today's bookings.
If the correlation is strongest at lag=14, customers take ~2 weeks to act on Reddit buzz.

In [ ]:
LAG_DAYS = [7, 14, 21, 28]

# Features to lag
FEATURES_TO_LAG = [
    "total_engagement_score",
    "total_weighted_sentiment",
    "mean_sentiment",
    "eng_score_roll7sum",
    "eng_score_roll14sum",
    "eng_score_roll28sum",
    "weighted_sent_roll7sum",
    "weighted_sent_roll14sum",
    "weighted_sent_roll28sum",
    "sentiment_momentum",
    "engagement_momentum",
    "engagement_zscore",
    "consistent_sentiment_score",
    "positive_days_7d",
    "positive_days_14d",
    "positive_days_28d",
    "positive_days_14d_detrended",
]

reddit_lagged = reddit_feat.sort(["lhg_country", "date"])

for lag in LAG_DAYS:
    reddit_lagged = reddit_lagged.with_columns([
        pl.col(feat)
          .shift(lag)          # shift forward: lag days into the future
          .over("lhg_country")
          .alias(f"{feat}_lag{lag}")
        for feat in FEATURES_TO_LAG
    ])

n_lag_cols = len(LAG_DAYS) * len(FEATURES_TO_LAG)
print(f"Added {n_lag_cols} lag features ({len(FEATURES_TO_LAG)} features × {len(LAG_DAYS)} lags)")
print(f"Reddit feature panel shape: {reddit_lagged.shape}")

## Join booking and Reddit panels

In [ ]:
# Inner join: only keep dates where BOTH booking and Reddit data exist
panel = bookings_feat.join(
    reddit_lagged,
    on=["date", "lhg_country"],
    how="inner"
)

print(f"Joined panel shape: {panel.shape}")
print(f"Date range        : {panel['date'].min()} → {panel['date'].max()}")
print(f"Countries         : {panel['lhg_country'].n_unique()}")
print()

#Check how many rows have nulls (from lag features at the start of each country's series)
null_counts = panel.null_count()
cols_with_nulls = {c: null_counts[c][0] for c in panel.columns if null_counts[c][0] > 0}
print(f"Columns with nulls (expected for lag features): {len(cols_with_nulls)}")
print("Max nulls in any column:", max(cols_with_nulls.values()) if cols_with_nulls else 0)

In [ ]:
# Null handling
# Lag features will have nulls at the START of each country's series
# (can't lag 28 days back when there's only 10 days of history)
# We drop rows where the longest lag (28d) is still null
# This removes the first 28 days per country — acceptable data loss

panel_clean = panel.filter(
    pl.col("total_engagement_score_lag28").is_not_null()
)

rows_dropped = len(panel) - len(panel_clean)
print(f"Rows dropped (lag warmup period): {rows_dropped:,}")
print(f"Final panel shape              : {panel_clean.shape}")
print(f"Date range                     : {panel_clean['date'].min()} → {panel_clean['date'].max()}")

## Feature overview

In [ ]:
# Feature summary statistics
key_features = [
    "pax", "pax_zscore", "pax_spike_positive",
    "total_engagement_score", "mean_sentiment",
    "sentiment_momentum", "engagement_momentum",
    "consistent_sentiment_score",
    "eng_score_roll7sum", "eng_score_roll28sum",
    "total_engagement_score_lag7",
    "total_engagement_score_lag14",
    "total_engagement_score_lag28",
]

print(panel_clean.select(key_features).describe())

In [ ]:
# Reddit engagement vs bookings for one country 
SAMPLE_COUNTRY = "DE"   # change to any country from panel

sample = (
    panel_clean
    .filter(pl.col("lhg_country") == SAMPLE_COUNTRY)
    .sort("date")
    .to_pandas()
    .set_index("date")
)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

#Bookings
axes[0].plot(sample["pax"], linewidth=0.8, color="steelblue")
axes[0].set_title(f"{SAMPLE_COUNTRY} — Daily bookings")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

#Reddit engagement
axes[1].plot(sample["eng_score_roll7sum"], linewidth=1, color="darkorange", label="7d rolling")
axes[1].plot(sample["eng_score_roll28sum"], linewidth=1.5, color="red", linestyle="--", label="28d rolling")
axes[1].set_title("Reddit engagement score (rolling sums)")
axes[1].legend(fontsize=9)

#Sentiment momentum
axes[2].plot(sample["sentiment_momentum"], linewidth=0.8, color="green")
axes[2].axhline(0, color="gray", linewidth=0.5, linestyle="--")
axes[2].set_title("Sentiment momentum (7d mean − 28d mean)")

plt.suptitle(f"Feature overview — {SAMPLE_COUNTRY}", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / f"features_{SAMPLE_COUNTRY}.png", dpi=150)
plt.show()

In [ ]:
# Complete feature list
print(f"Total columns in panel: {len(panel_clean.columns)}")
print()
print("Booking features:")
print([c for c in panel_clean.columns if "pax" in c])
print()
print("Reddit base features:")
print([c for c in panel_clean.columns
       if any(x in c for x in ["engagement", "sentiment", "n_posts", "positive_days"])
       and "lag" not in c])
print()
print("Lag features (sample):")
print([c for c in panel_clean.columns if "lag" in c][:10], "...")

## Save Outputs

In [ ]:
panel_clean.write_parquet(OUTPUT_PATH / "panel_features.parquet")

KEY_COLS = (
    ["date", "lhg_country", "pax", "pax_zscore", "pax_spike_positive",
     "pax_roll7sum", "pax_roll14sum", "pax_roll28sum"] +
    [c for c in panel_clean.columns if "lag" in c] +
    ["sentiment_momentum", "engagement_momentum", "engagement_zscore",
     "consistent_sentiment_score", "positive_days_7d",
     "positive_days_14d", "positive_days_28d"]
)
panel_clean.select(KEY_COLS).write_parquet(OUTPUT_PATH / "panel_features_slim.parquet")

print("Saved:")
print(f"  outputs/panel_features.parquet")
print(f"  outputs/panel_features_slim.parquet")